# Reproducing Chandravadia et al. (2020) from dandiset 000004

**Paper.** Chandravadia N, Liang D, Schjetnan AGP, Carlson A, Faraut M, Chung JM, Reed CM, Dichter B,
Maoz U, Kalia SK, Valiante TA, Mamelak AN, Rutishauser U (2020). A NWB-based dataset and processing
pipeline of human single-neuron activity during a declarative memory task. *Scientific Data* 7, 78.
[doi:10.1038/s41597-020-0415-9](https://doi.org/10.1038/s41597-020-0415-9). The article is open
access (CC BY 4.0); this attempt used its full text, including Tables 1 and 2 and Online-only Table 1.

**Dandiset.** [DANDI:000004](https://dandiarchive.org/dandiset/000004/0.220126.1852), version
0.220126.1852. The paper cites no DANDI version: its data statement points to the Open Science
Framework (doi:10.17605/OSF.IO/HV7JA), and the dandiset was created on 2020-03-16, after the paper
appeared. The archive has two published versions, 0.220126.1851 and 0.220126.1852, published a
minute apart with the same 87 assets (checked in the first data cell) and metadata that differs only
in version fields. The most recent is pinned. The draft holds the same files.

**What the dandiset holds.** One NWB file per session: 87 sessions of 59 subjects, with the trials
table, the TTL event stream, the stimulus images, the spike-sorted units with mean waveforms, and
the electrodes table. There are no raw broadband recordings.

**Code read.** [rutishauserlab/recogmem-release-NWB](https://github.com/rutishauserlab/recogmem-release-NWB)
at commit 1d95162 (2025-01-14): the Python and MATLAB export and analysis code, the session list
`defineNOsessions_release.ini`, and `plotFigure5.ipynb`. The code settles what the Methods leave
open (the AUC, the z-ROC, the high/low confidence split, the label coding). Where it disagrees with
the Methods (the analysis window, and correct trials only for memory selectivity), the Methods give
the main result and the code's choice is shown as a variant. `plotFigure5.ipynb` writes the Figure
5f percentages into the plot as constants, so the code does not show how they were computed.

**Outside inputs.** Online-only Table 1 of the paper is copied into the helper module as text, to
compare with the files' subject metadata. Table 1 of Faraut et al. 2018 (Scientific Data 5, 180010,
doi:10.1038/sdata.2018.10), the paper describing the earlier release, is copied the same way; it
lists that release's sessions and is used only to mark which sessions of the dandiset are new
(previous-release-link and previous-release-additions).

| Claim | Status |
|---|---|
| counts-neurons-subjects: 1863 neurons, 59 subjects | tested |
| counts-sessions: 87 sessions of Online-only Table 1 | tested |
| table1-demographics: age and sex of every subject | tested |
| task-structure: 100 encoding and 100 recognition trials, 5 categories | tested |
| new-old-label-coding: label 0 is old, 1 is new | tested |
| ttl-codes: event codes of Table 1 | tested |
| experiment-ids: block ids of Table 2 | tested |
| session-dates: day set to the first of the month | tested |
| units-table: columns, 100 kHz waveforms, one electrode per unit | tested |
| electrodes-table: area, coordinates, impedance, filter, channel | tested |
| isi-criterion: fewer than 3% of ISIs below 3 ms | tested |
| fig5a-mtl-locations: all units in amygdala or hippocampus | tested |
| fig5c-auc-all: AUC 0.74 ± 0.10 | tested |
| fig5c-auc-sites: AUC 0.74 ± 0.09 (CS) and 0.69 ± 0.13 (TWH) | tested |
| fig5b-roc-asymmetric: asymmetric ROC | tested |
| fig5d-confidence-accuracy: high confidence more accurate | tested |
| vs-cells-proportion: 343 VS cells (18.4%) | tested |
| ms-cells-proportion: 146 MS cells (7.8%) | tested |
| dual-cells-proportion: 26 dual cells (1.4%) | tested |
| fig5f-site-proportions: comparable proportions at CS and TWH | tested |
| twh-sessions: 9 TWH sessions | tested |
| fig4-example-vs-neuron: neuron 132-177-4-RA | tested |
| previous-release-link: earlier release's sessions found here | tested (with outside input) |
| previous-release-additions: 23 sessions, 17 subjects, 288 neurons added | tested (with outside input) |
| fig3-example-vs-neuron: neuron 144-140-3-RH | blocked (data missing) |

In [ ]:
import time
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import recogmem as rm

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = rm.load_all()
by_id = {s["session"]: s for s in sessions}
print(f"loaded {len(sessions)} files from dandiset {rm.DANDISET_ID} version {rm.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "session": s["session"], "label": s["label"], "subject": s["subject_id"], "site": s["site"],
    "institution": s["institution"], "start": s["session_start_time"][:10], "age": s["age"], "sex": s["sex"],
    "trials": len(s["trials"]), "learn": int((s["trials"].stim_phase == "learn").sum()),
    "recog": int((s["trials"].stim_phase == "recog").sum()), "units": len(s["spikes"]),
    "electrodes": len(s["electrodes"]), "categories": s["trials"].stimCategory.nunique(),
    "experiment_ids": tuple(int(x) for x in np.unique(s["experiment_ids"])),
    "data_collection": s["data_collection"]} for s in sessions])
print("Files:", len(info), " subjects:", info.subject.nunique(), " species:", sorted({s["species"] for s in sessions}))
print("\nBy site (CS = subjects labelled H or C, TWH = labelled T):")
print(info.groupby("site").agg(sessions=("session", "size"), subjects=("subject", "nunique"), units=("units", "sum")).to_string())
print("\nInstitution strings:", info.institution.value_counts().to_dict())
print("Devices:", pd.Series([d for s in sessions for d in s["devices"]]).value_counts().to_dict())
print("\nTrials per file (learn, recog):", info.groupby(["learn", "recog"]).size().to_dict())
print("Units per file: min", info.units.min(), "median", info.units.median(), "max", info.units.max(), "total", info.units.sum())
print("Is there a session 144 (Figure 3 example)?", 144 in by_id)
client = rm.DandiAPIClient()
versions = {}
for v in ("0.220126.1851", "0.220126.1852", "draft"):
    versions[v] = {a.path: (a.identifier, a.size) for a in client.get_dandiset(rm.DANDISET_ID, v).get_assets()}
print("Published versions 0.220126.1851 and 0.220126.1852 hold the same assets (ids and sizes):", versions["0.220126.1851"] == versions["0.220126.1852"])
print("The draft holds the same paths and sizes as the pinned version:",
      {p: s for p, (_, s) in versions["draft"].items()} == {p: s for p, (_, s) in versions["0.220126.1852"].items()})
print()
print(info.drop(columns=["institution", "data_collection"]).to_string(index=False))

The archive's summary gives 59 subjects and 87 files, which the files confirm, and the two
published versions hold the same assets. There are 1864 unit rows, one more than the paper's 1863.
Eight files have fewer than 100 learning trials (seven with 50 and one with 97). The institution
is written "Hunigton Memorial Hospital" in 28 files and "Hunington Memorial Hospital" in 6, and
the device both "Neuralynx-cheetah" and "Neuralynx-Cheetah". There is no session 144, the session
of the example neuron of Figure 3.

In [ ]:
# Units and coding of the variables used: times, responses, labels, categories.
rows = []
for s in sessions:
    t = s["trials"]
    r = rm.recognition(s)
    ev, et = s["events"], s["events_t"]
    stim_on = et[ev == 1]
    on_idx = np.flatnonzero(ev == 1)
    resp_ev = np.array([ev[i + 3] if i + 3 < len(ev) else np.nan for i in on_idx])
    sp = np.concatenate(s["spikes"])
    rows.append({
        "session": s["session"],
        "stim_s_median": np.median(t.stim_off_time - t.stim_on_time),
        "start_is_stim_on": bool(np.allclose(t.start_time, t.stim_on_time)),
        "onsets_match_events": bool(len(stim_on) == len(t) and np.allclose(np.sort(stim_on), np.sort(t.stim_on_time.values))),
        "resp_mismatch": int(np.sum(resp_ev != np.where(t.stim_phase == "recog", t.response_value, t.response_value + 20)))
                         if len(resp_ev) == len(t) else -1,
        "unlisted_after_question": sorted({int(x) for x in resp_ev} - set(range(20, 22)) - set(range(31, 37))),
        "resp_min": int(r.response.min()), "resp_max": int(r.response.max()),
        "labels": tuple(sorted(r.label.unique())),
        "spikes_from_s": sp.min() - t.start_time.min(), "spikes_to_s": sp.max() - t.stop_time.max(),
        "trial_s_median": np.median(t.stop_time - t.start_time)})
units = pd.DataFrame(rows)
print("Stimulus duration (s), median per file: range", units.stim_s_median.min().round(3), "to", units.stim_s_median.max().round(3))
print("Trial duration, stimulus onset to end of trial (s), median per file:", units.trial_s_median.min().round(2), "to", units.trial_s_median.max().round(2))
print("start_time equals stim_on_time in every file:", units.start_is_stim_on.all())
print("Trial onsets equal the stimulus-onset events (TTL 1) in every file:", units.onsets_match_events.all())
print("Per trial, the event three after stimulus onset (the response the paper's MATLAB code reads) against response_value")
print("(recognition: 31-36; learning: response_value + 20 against 20/21): files with every trial equal:", int((units.resp_mismatch == 0).sum()), "of", len(units))
print(units[units.resp_mismatch != 0][["session", "resp_mismatch", "unlisted_after_question"]].to_string(index=False))
print("Recognition responses (response_value - 30) span", units.resp_min.min(), "to", units.resp_max.max(), "; labels:", sorted({x for l in units.labels for x in l}))
print("First spike relative to first trial onset (s): range", units.spikes_from_s.min().round(1), "to", units.spikes_from_s.max().round(1))
print("Last spike relative to last trial end (s): range", units.spikes_to_s.min().round(1), "to", units.spikes_to_s.max().round(1))
print("Files whose first spike is more than 60 s before the first trial:", len(units[units.spikes_from_s < -60]), units[units.spikes_from_s < -60].session.tolist())
print("\nDescription of new_old_labels_recog in the files:", repr(sessions[0]["trial_descriptions"]["new_old_labels_recog"]))
print("Learning response_value values:", sorted(set(np.concatenate([s["trials"][s["trials"].stim_phase == "learn"].response_value.unique() for s in sessions]))))

Times are in seconds on one clock per file: trial start times equal the stimulus-onset events and
stimuli last a median of 1.0 to 2.0 s per file (the Methods give no duration). In 81 of 87 files the
trials table's response_value equals, trial by trial, the response event the paper's MATLAB code
reads; in the six others 1 to 18 trials differ, where extra codes (3 or 23) or a shifted sequence
follow the question. The analysis takes responses from response_value minus 30, which spans 1 to 6.
In learning trials the table stores 0 and 1 for the events 20 and 21. The column description of
new_old_labels_recog says 0 is old; this is tested under new-old-label-coding. In 15 files spikes
start more than 60 s before the first trial (up to 2462 s), so the units cover more than the task.

In [ ]:
# Recordings deposited more than once: identical spike trains, trial times or event sequences.
pairs = []
for a, b in combinations(sessions, 2):
    same_spikes = len(a["spikes"]) == len(b["spikes"]) and all(
        len(x) == len(y) and np.array_equal(x, y) for x, y in zip(a["spikes"], b["spikes"]))
    ta, tb = a["trials"], b["trials"]
    n = min(len(ta), len(tb))
    shared_trials = int(np.sum(np.isin(np.round(ta.start_time.values, 4), np.round(tb.start_time.values, 4))))
    if same_spikes or shared_trials >= 10:
        pairs.append({"a": a["session"], "b": b["session"], "units": len(a["spikes"]), "identical_spike_trains": same_spikes,
                      "shared_trial_onsets": shared_trials, "trials_a": len(ta), "trials_b": len(tb),
                      "exp_ids_a": tuple(int(x) for x in np.unique(a["experiment_ids"])),
                      "exp_ids_b": tuple(int(x) for x in np.unique(b["experiment_ids"])),
                      "a_trials_s": (round(ta.start_time.min()), round(ta.stop_time.max())),
                      "b_trials_s": (round(tb.start_time.min()), round(tb.stop_time.max())),
                      "spikes_s": (round(np.concatenate(a["spikes"]).min()), round(np.concatenate(a["spikes"]).max()))})
pairs = pd.DataFrame(pairs)
print(pairs.to_string(index=False))
DUPLICATE_SECOND = sorted(pairs[pairs.identical_spike_trains].b.tolist())
print("\nSessions whose units repeat another session's (dropped when each recording is counted once):", DUPLICATE_SECOND)
print("Units in them:", sum(len(by_id[k]["spikes"]) for k in DUPLICATE_SECOND))
same_pair = pairs[(pairs.a == 54) & (pairs.b == 55)]
r54, r55 = rm.recognition(by_id[54]), rm.recognition(by_id[55])
l54 = by_id[54]["trials"][by_id[54]["trials"].stim_phase == "learn"]
l55 = by_id[55]["trials"][by_id[55]["trials"].stim_phase == "learn"]
print("Sessions 54 and 55: learning trials identical:", l54.reset_index(drop=True).equals(l55.reset_index(drop=True)),
      "; recognition onsets shared:", int(np.isin(r54.start_time.round(4), r55.start_time.round(4)).sum()),
      "; recognition images shared:", len(set(r54.external_image_file) & set(r55.external_image_file)))

Two pairs of files hold identical spike trains for every unit. Sessions 43 and 44 (subject H23)
are two different task blocks of one continuous recording: the spike trains span both blocks,
session 44's trials come first and session 43's later. Sessions 54 and 55 (subject H42) share
the same learning block, trial for trial, and differ in their recognition blocks (experiment ids
81 and 82), so session 55 is a second recognition test of the same encoding block, deposited with
a copy of the learning trials and the same spike trains. In both pairs the units are the same
neurons. The paper counts neurons per session, so its 1863 includes them twice; results are also
given with sessions 44 and 55 dropped.

In [ ]:
# Clock check: category selectivity of the units in 1 s windows shifted around stimulus onset, with
# the true labels and with the labels of the previous trial. A response to the stimulus can only
# appear after onset and with the true labels; an offset between spikes and trials would move it.
offsets = np.arange(-1.6, 1.61, 0.2)
frac_true, frac_shift = [], []
per_unit = []
for s in sessions:
    r = rm.recognition(s)
    on, cat = r.start_time.values, r.category.values
    for k, sp in enumerate(s["spikes"]):
        p_true = [rm.anova_p(rm.window_rates(sp, on, (o, o + 1.0)), cat) for o in offsets]
        p_shift = [rm.anova_p(rm.window_rates(sp, on, (o, o + 1.0)), np.roll(cat, 1)) for o in offsets]
        per_unit.append({"session": s["session"], "p_true": p_true, "p_shift": p_shift,
                         "p_pre": rm.anova_p(rm.window_rates(sp, on, (-1.0, 0.0)), cat),
                         "p_post": rm.anova_p(rm.window_rates(sp, on, rm.WINDOW_METHODS), cat)})
P = np.array([u["p_true"] for u in per_unit]); PS = np.array([u["p_shift"] for u in per_unit])
clock = pd.DataFrame({"window_start_s": offsets.round(1), "frac_p<0.05_true_labels": (P < 0.05).mean(0).round(3),
                      "frac_p<0.05_previous_trial_labels": (PS < 0.05).mean(0).round(3)})
print(clock.to_string(index=False))
pu = pd.DataFrame(per_unit)
ses = pu.groupby("session").agg(units=("p_post", "size"), sig_pre=("p_pre", lambda p: int((p < 0.05).sum())),
                                sig_post=("p_post", lambda p: int((p < 0.05).sum())))
ses["binom_p_post_vs_5pct"] = [stats.binomtest(k, n, 0.05, alternative="greater").pvalue for k, n in zip(ses.sig_post, ses.units)]
ses["established"] = (ses.binom_p_post_vs_5pct < 0.05) & (ses.sig_post > ses.sig_pre)
print("\nSessions where the stimulus response (more category-selective units after onset than expected by chance,")
print("and more than before onset) establishes the shared clock:", int(ses.established.sum()), "of", len(ses))
print("Not established (too few units or no excess):", ses[~ses.established].index.tolist())
print("Units in sessions not established:", int(ses[~ses.established].units.sum()))
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(offsets, (P < 0.05).mean(0), "o-", label="true labels")
ax.plot(offsets, (PS < 0.05).mean(0), "s-", label="previous trial's labels")
ax.axhline(0.05, color="k", lw=0.8)
ax.set_xlabel("window start relative to stimulus onset (s), 1 s windows")
ax.set_ylabel("fraction of units p < 0.05")
ax.legend(); plt.tight_layout(); plt.show()
lo_, hi_ = min((P < 0.05).mean(0).min(), (PS < 0.05).mean(0).min()), max((P < 0.05).mean(0).max(), (PS < 0.05).mean(0).max())
print(f"Plotted fractions range from {lo_:.3f} to {hi_:.3f}; none outside the axes.")

With the true labels, the fraction of category-selective units is 4.1% to 4.9% for windows that end
at or before stimulus onset (start at -1.0 s or earlier), rises once the window reaches past 200 ms
after onset, peaks at 17.8% for the window starting at 0.2 s and declines after. With the labels of
the previous trial it stays between 3.9% and 6.0% at every offset. Spikes and trials therefore share
a clock overall. Per session this is shown for 38 of 87 sessions; the 49 others, listed above with
801 units, have too few units or too few selective ones to tell, and for them the shared clock is
consistent with the data but assumed.

## counts-neurons-subjects

> The dataset holds 1863 single neurons recorded from the medial temporal lobes of 59 subjects.

In [ ]:
n_units = int(info.units.sum())
n_units_once = int(info[~info.session.isin(DUPLICATE_SECOND)].units.sum())
n_subjects = info.subject.nunique()
print(f"(a) every unit row: {n_units} neurons (paper 1863, tolerance 1676.7 to 2049.3; difference {n_units - 1863:+d})")
print(f"(b) duplicated spike trains counted once: {n_units_once} neurons (difference {n_units_once - 1863:+d})")
print(f"subjects: {n_subjects} (paper 59)")
print("units by site:", info.groupby("site").units.sum().to_dict())
print("\nunits per site:", info.groupby("site").units.sum().to_dict())
print("units per session:")
print(info.set_index("session").units.to_frame().T.to_string())
fewest = sorted(((len(sp), s["session"], k) for s in sessions for k, sp in enumerate(s["spikes"])))[:5]
print("five units with the fewest spikes (spikes, session, unit):", fewest)

**Result: reproduced.** The files hold 1864 unit rows, one more than the paper's 1863, and 59
subjects, both within tolerance. Counting the 55 units of sessions 44 and 55 once, as their spike
trains repeat those of sessions 43 and 54, gives 1809, also within tolerance. The
files hold 1787 units in CS sessions and 77 in TWH sessions. The paper gives no count per session or
site, so the extra unit cannot be identified; no unit is empty (the fewest spikes is 22).

## counts-sessions

> The patients performed 87 sessions, listed by subject and session id in Online-only Table 1
> (the Methods give 89).

In [ ]:
table = rm.online_table_1()
files_set = set(zip(info.label.str.replace("CS", "C").str.replace(r"^([A-Z])0*", r"\1", regex=True), info.session))
table_set = set(zip(table.table_subject.str.replace(r"^([A-Z])0*", r"\1", regex=True), table.session))
print("sessions in the files:", len(info), "; in Online-only Table 1:", len(table), "; Methods: 89")
print("(subject, session) pairs only in the files:", sorted(files_set - table_set))
print("(subject, session) pairs only in the table:", sorted(table_set - files_set))
print("Task variant against the stimulus set of the file:")
var = table.set_index("session").variant
print(pd.crosstab(info.session.map(var).rename("variant (table)"),
                  pd.Series([sorted(s["trials"].category_name.unique())[0] for s in sessions], name="first category name")).to_string())

**Result: reproduced.** The 87 files match the 87 (subject, session) pairs of Online-only Table 1
exactly, with none missing or extra; the Methods' 89 sessions disagree with the paper's own table.
Each table variant corresponds to one stimulus set in the files.

## table1-demographics

> Online-only Table 1 gives the age and sex of every subject; there are 25 female and 34 male
> subjects.

In [ ]:
subj = info.groupby("subject").agg(label=("label", "first"), age=("age", "first"), sex=("sex", "first"),
                                   ages=("age", "nunique"), sexes=("sex", "nunique")).reset_index()
subj["session"] = info.groupby("subject").session.first().values
m = subj.merge(table[["session", "table_subject", "age", "sex", "diagnosis"]], on="session", suffixes=("_file", "_table"))
m["age_file"] = m.age_file.astype(int)
m["match"] = (m.age_file == m.age_table) & (m.sex_file == m.sex_table)
diag = {s["subject_id"]: s["diagnosis"] for s in sessions}
m["diagnosis_file"] = m.subject.map(diag)
print("Subjects with one age and one sex across their files:", int(((subj.ages == 1) & (subj.sexes == 1)).sum()), "of", len(subj))
print("Subjects matching the table in age and sex:", int(m.match.sum()), "of", len(m))
print(m[~m.match][["subject", "table_subject", "age_file", "age_table", "sex_file", "sex_table"]].to_string(index=False))
print("\nFemale / male in the files:", subj.sex.value_counts().to_dict(), "; in the table: F 25, M 34")
print("\nDiagnosis, files against table, where the text differs:")
print(m[m.diagnosis_file.str.lower() != m.diagnosis.str.lower()][["table_subject", "diagnosis_file", "diagnosis"]].to_string(index=False))

**Result: partially reproduced.** Age and sex match the table for 56 of 59 subjects. C24 is 57 in
the files and 47 in the table, H33 is 30 and 29, and H48 is a 32-year-old woman in the files and a
54-year-old man in the table. The files hold 26 female and 33 male subjects, within tolerance of the
table's 25 and 34. The diagnosis is coded more coarsely in the files for most subjects. It
differs in substance for C61 (Left Lateral Frontal against Left Mesial Temporal); for C56 and H51
the files leave out part of the table's diagnosis (orbitofrontal, temporal), and for T90 they add
"Bilateral".

## task-structure

> Each session has 100 encoding trials showing 100 novel images and 100 recognition trials, of
> which 50 show new and 50 old images, and the images come from 5 visual categories.

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]; learn = t[t.stim_phase == "learn"]; r = rm.recognition(s)
    (shape, img_t), = s["stimulus"].values()
    rows.append({"session": s["session"], "learn": len(learn), "learn_distinct_images": learn.external_image_file.nunique(),
                 "recog": len(r), "label_counts": tuple(r.label.value_counts().sort_index()), "categories": t.stimCategory.nunique(),
                 "category_names": t.category_name.nunique(), "images_stored": shape[0],
                 "image_times_are_onsets": bool(np.allclose(np.sort(img_t), np.sort(t.start_time.values)))})
task = pd.DataFrame(rows)
ok_learn = (task.learn == 100) & (task.learn_distinct_images == 100)
ok_recog = (task.recog == 100) & (task.label_counts == (50, 50))
ok_cat = (task.categories == 5) & (task.category_names == 5)
print("100 encoding trials with 100 distinct images:", int(ok_learn.sum()), "of", len(task))
print("100 recognition trials, 50 per label:", int(ok_recog.sum()), "of", len(task))
print("5 categories (ids and names):", int(ok_cat.sum()), "of", len(task))
print("One stored image per trial, timestamped at trial onset:", int(((task.images_stored == task.learn + task.recog) & task.image_times_are_onsets).sum()), "of", len(task))
print("\nSessions with other encoding blocks:")
print(task[~ok_learn].to_string(index=False))
print("\nStimulus storage: one ImageSeries per file,", sorted({k for s in sessions for k in s["stimulus"]}),
      "(the paper describes stimuli_learn_x and stimuli_recog_x entries)")

**Result: partially reproduced.** All 87 sessions have 100 recognition trials with 50 of each
label and 5 categories, and one stored image per trial. The encoding block has 100 trials with 100
distinct images in 79 sessions; seven sessions have 50 encoding trials and session 123 has 97. The
images are stored as one image series per file, not as the stimuli_learn_x and stimuli_recog_x
entries the paper describes. In the
eight sessions with shorter encoding blocks, 2% to 48% of the images shown as old in recognition
have no encoding trial in the file (shown under new-old-label-coding), so the file does not hold
the encoding of every old image.

## new-old-label-coding

> In the trials table, new_old_labels_recog is 0 for old stimuli (seen during encoding) and 1 for
> new stimuli.

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]; r = rm.recognition(s)
    learned = set(t[t.stim_phase == "learn"].external_image_file)
    seen = r.external_image_file.isin(learned)
    rows.append({"session": s["session"],
                 "label0_seen_in_encoding": seen[r.label == 0].mean(), "label1_seen_in_encoding": seen[r.label == 1].mean(),
                 "label0_old_responses": (r.response[r.label == 0] >= 4).mean(), "label1_old_responses": (r.response[r.label == 1] >= 4).mean()})
lab = pd.DataFrame(rows)
print("Fraction of recognition trials whose image file was shown during encoding, median over sessions:")
print("  label 0:", lab.label0_seen_in_encoding.median(), " label 1:", lab.label1_seen_in_encoding.median())
print("Sessions where every label-1 image and no label-0 image was shown in encoding:",
      int(((lab.label1_seen_in_encoding == 1) & (lab.label0_seen_in_encoding == 0)).sum()), "of", len(lab))
print("Sessions where label 0 matches the stated coding (all label-0 images seen, no label-1 image seen):",
      int(((lab.label0_seen_in_encoding == 1) & (lab.label1_seen_in_encoding == 0)).sum()))
print("\nFraction of 'old' responses (4-6), median over sessions: label 0", round(lab.label0_old_responses.median(), 3),
      " label 1", round(lab.label1_old_responses.median(), 3))
print("Sessions with more 'old' responses to label 1 than to label 0:", int((lab.label1_old_responses > lab.label0_old_responses).sum()), "of", len(lab))
print("Sign test, label 1 above label 0: p =", stats.binomtest(int((lab.label1_old_responses > lab.label0_old_responses).sum()),
      int((lab.label1_old_responses != lab.label0_old_responses).sum())).pvalue)
print("Sessions with imperfect image match:")
print(lab[(lab.label1_seen_in_encoding < 1) | (lab.label0_seen_in_encoding > 0)].round(3).to_string(index=False))

**Result: not reproduced.** The coding is the reverse of the stated one. In 79 of 87 sessions every
label-1 image and no label-0 image was shown during encoding, and in no session does label 0 mark
the encoded images. Label-1 trials draw "old" responses more often than label-0 trials in 86 of 87
sessions (median 0.70 against 0.24, sign test p = 1.1e-24). Label 1 is old, as the paper's code
reads it; the paper's text and the files' column description say 0. In the eight sessions with
shortened encoding blocks only 52% to 98% of the label-1 images are among the stored encoding
trials, so encoding trials are missing from those files.

## ttl-codes

> The events stream holds the TTL markers 55, 1, 2, 3, 20 or 21, 31 to 36, 6 and 66 with the
> meanings of Table 1, and the events and experiment_ids streams have one entry each, in the same
> order.

In [ ]:
LISTED = {55, 1, 2, 3, 20, 21, 31, 32, 33, 34, 35, 36, 6, 66}
rows = []
for s in sessions:
    ev, et = s["events"], s["events_t"]
    same = len(ev) == len(s["experiment_ids"]) and np.allclose(et, s["experiment_ids_t"])
    other = sorted(set(ev.astype(int)) - LISTED)
    on = np.flatnonzero(ev == 1)
    t = s["trials"]
    phase = t.stim_phase.values[np.abs(t.start_time.values[None, :] - et[on][:, None]).argmin(1)]
    seq_ok = phase_ok = 0
    for i, ph in zip(on, phase):
        nxt = ev[i + 1:i + 5]
        if len(nxt) == 4 and nxt[0] == 2 and nxt[1] == 3 and (nxt[2] in (20, 21) or 31 <= nxt[2] <= 36) and nxt[3] == 6:
            seq_ok += 1
            phase_ok += (nxt[2] in (20, 21)) if ph == "learn" else (31 <= nxt[2] <= 36)
    rows.append({"session": s["session"], "events": len(ev), "same_length_and_times": same, "unlisted_codes": tuple(other),
                 "trials": len(on), "trials_in_order": seq_ok, "response_kind_of_phase": phase_ok,
                 "starts_55": int((ev == 55).sum()), "ends_66": int((ev == 66).sum())})
ttl = pd.DataFrame(rows)
print("events and experiment_ids of equal length and timestamps:", int(ttl.same_length_and_times.sum()), "of", len(ttl))
print("files with codes not in Table 1:", ttl[ttl.unlisted_codes != ()][["session", "unlisted_codes"]].to_string(index=False))
print("trials showing 1, 2, 3, response, 6 in order:", int(ttl.trials_in_order.sum()), "of", int(ttl.trials.sum()))
print("trials in order whose response is of the phase's kind (20 or 21 in learning, 31 to 36 in recognition):",
      int(ttl.response_kind_of_phase.sum()), "of", int(ttl.trials_in_order.sum()))
print("(start 55, end 66) markers per file: number of files", ttl.groupby(["starts_55", "ends_66"]).size().to_dict())
print("files without a 55:", ttl[ttl.starts_55 == 0].session.tolist(), "; files with other than two 66:", ttl[ttl.ends_66 != 2].session.tolist())
ttl["all_ok"] = (ttl.same_length_and_times & (ttl.unlisted_codes == ()) & (ttl.trials_in_order == ttl.trials)
                 & (ttl.response_kind_of_phase == ttl.trials_in_order))
print("files meeting every condition:", int(ttl.all_ok.sum()), "of", len(ttl))
print(ttl[~ttl.all_ok].to_string(index=False))

**Result: partially reproduced.** In all 87 files the events and experiment_ids streams have the
same length and timestamps, and 17019 of 17047 trials show onset, offset, question, response and end
in order. 78 of 87 files meet every condition; nine hold codes Table 1 does not list (0, 4, 22, 23,
98 or 99), seven files (sessions 92, 93, 96, 97, 99, 100 and 101) have no start-of-experiment
code 55, and session 2001 has three end codes 66. In every trial that shows the sequence, the response
is of its phase's kind (20 or 21 in learning, 31 to 36 in recognition).

## experiment-ids

> The learning block is labelled with one of 80, 83 or 88 and the recognition block with one of
> 81, 84 or 89.

In [ ]:
rows = []
for s in sessions:
    ids = s["experiment_ids"]; ev = s["events"]
    first_resp = {int(e): int(ev[(ids == e) & (((ev >= 20) & (ev <= 21)) | ((ev >= 31) & (ev <= 36)))][0]) for e in np.unique(ids)}
    learn_id = [e for e, r in first_resp.items() if r in (20, 21)]
    recog_id = [e for e, r in first_resp.items() if 31 <= r <= 36]
    rows.append({"session": s["session"], "learn_id": learn_id[0] if learn_id else None,
                 "recog_id": recog_id[0] if recog_id else None, "data_collection": s["data_collection"]})
eid = pd.DataFrame(rows)
eid["ok"] = eid.learn_id.isin([80, 83, 88]) & eid.recog_id.isin([81, 84, 89])
print("Block identified by its responses (20/21 learning, 31-36 recognition).")
print("files whose ids are in the stated sets:", int(eid.ok.sum()), "of", len(eid))
print(eid[~eid.ok].to_string(index=False))
print("\nid pairs:", eid.groupby(["learn_id", "recog_id"]).size().to_dict())

**Result: partially reproduced.** 83 of 87 files use the ids of Table 2. Session 55 labels its
recognition block 82, session 77 uses 81 for learning and 82 for recognition, and sessions 100 and
105 use 82 for learning and 83 for recognition.

## session-dates

> session_start_time keeps only the year and month of the experiment, with the day set to the
> first of the month for all sessions.

In [ ]:
dates = pd.DataFrame({"session": [s["session"] for s in sessions], "start": [s["session_start_time"] for s in sessions]})
dates["day"] = dates.start.str[8:10]; dates["year"] = dates.start.str[:4].astype(int)
print("files on the first of a month:", int((dates.day == "01").sum()), "of", len(dates))
print(dates[dates.day != "01"].to_string(index=False))
print("years:", dates.year.min(), "to", dates.year.max(), "; by year:", dates.year.value_counts().sort_index().to_dict())

**Result: partially reproduced.** 86 of 87 files start on the first of a month. Session 48 starts on
2010-03-13, a day that was not set to the first. Session 68 is dated 2000-01-01, outside the span
of the other recordings (2006 to 2019).

## units-table

> The units table holds spike_times, waveform_mean_encoding, waveform_mean_recognition (sampled
> at 100 kHz), origClusterID, SNR, IsolDist and electrodes, with one electrode per neuron.

In [ ]:
needed = {"spike_times", "waveform_mean_encoding", "waveform_mean_recognition", "origClusterID", "SNR", "IsolationDist", "electrodes"}
cols_ok = sum(needed <= set(s["unit_columns"]) for s in sessions)
one_elec = sum(bool(np.array_equal(s["unit_electrode_index"], np.arange(1, len(s["spikes"]) + 1))) for s in sessions)
rates = np.unique(np.concatenate([s["waveform_rate"] for s in sessions]))
snr = np.concatenate([s["snr"] for s in sessions]); iso = np.concatenate([s["isolation_distance"] for s in sessions])
print("files with every column (IsolDist is named IsolationDist):", cols_ok, "of", len(sessions))
print("columns in every file:", sorted(set.intersection(*[set(s["unit_columns"]) for s in sessions])))
print("files where every unit links to exactly one electrode:", one_elec, "of", len(sessions))
print("stated waveform sampling rate (Hz):", rates, "; waveform samples:", sorted({s["waveform_shape"][1] for s in sessions}))
print(f"SNR: {np.isfinite(snr).sum()} of {len(snr)} finite; {np.isinf(snr).sum()} infinite; finite range {np.nanmin(snr[np.isfinite(snr)]):.2f} to {np.nanmax(snr[np.isfinite(snr)]):.2f}")
print(f"IsolationDist: {np.isfinite(iso).sum()} of {len(iso)} finite; {np.isnan(iso).sum()} NaN; finite range {np.nanmin(iso):.3g} to {np.nanmax(iso[np.isfinite(iso)]):.3g}; above 1e4: {(iso > 1e4).sum()}")
unsorted = [(s["session"], k) for s in sessions for k, sp in enumerate(s["spikes"]) if np.any(np.diff(sp) < 0)]
print("units whose spike times are not in increasing order:", len(unsorted), "in sessions", sorted({a for a, _ in unsorted}))
for a, k in unsorted:
    sp = by_id[a]["spikes"][k]
    n_dup = int(np.sum(np.diff(np.sort(sp)) == 0))
    if n_dup:
        print(f"  session {a} unit {k}: {len(sp)} spike times, {n_dup} of them repeat another exactly")

**Result: partially reproduced.** Every file has every column (IsolDist is named IsolationDist),
and every unit links to one electrode. The stated waveform sampling rate is 98.4 kHz, within the
tolerance of the paper's 100 kHz. The quality metrics are not finite for every unit: SNR is finite
for 1778 of 1864 units, 11 being infinite, and the isolation distance is NaN for 493 units, with 4
values above 1e4 (up to 8.08e+32). The spike times of 13 units in 7 sessions are not in increasing
order, and two units of session 58 hold every spike time twice.

## electrodes-table

> The electrodes table gives each electrode's brain area, MNI coordinates, impedance, the filter
> applied before spike sorting (300 to 3000 Hz) and the original hardware channel.

In [ ]:
E = pd.concat([s["electrodes"].assign(session=s["session"]) for s in sessions], ignore_index=True)
print("electrodes:", len(E), "; columns in every file:", sorted(set.intersection(*[set(s["electrodes"].columns) for s in sessions])))
for c in ["location", "x", "y", "z", "imp", "filtering", "origChannel"]:
    v = E[c]
    empty = v.isna() | (v.astype(str).str.strip() == "")
    print(f"{c:12s} empty for {int(empty.sum())} of {len(E)}; files with any empty: {E[empty].session.nunique()}")
print("filtering values:", E.filtering.value_counts().to_dict())
print("sessions with missing coordinates:", sorted(E[E.x.isna()].session.unique()))
print("coordinate ranges (mm): x", E.x.min(), "to", E.x.max(), "; y", E.y.min(), "to", E.y.max(), "; z", E.z.min(), "to", E.z.max())
side = E.dropna(subset=["x"])
print("left-named electrodes with x < 0:", int((side.location.str.startswith("Left") & (side.x < 0)).sum()), "of", int(side.location.str.startswith("Left").sum()),
      "; right-named with x > 0:", int((side.location.str.startswith("Right") & (side.x > 0)).sum()), "of", int(side.location.str.startswith("Right").sum()))
print("imp: dtype", E.imp.dtype, "; distinct stored values:", [repr(v) for v in pd.unique(E.imp)][:10])

**Result: not reproduced.** Location, filter (300-3000Hz) and original channel are filled for all 887
electrodes, and the coordinates match the hemisphere of the location wherever they are given. The
impedance is stored as NaN (float64) for all 887 electrodes, and the coordinates are missing for 99 electrodes in 8
sessions.

## isi-criterion

> Clusters were accepted as units only when fewer than 3% of their interspike intervals were
> shorter than 3 ms.

In [ ]:
isi = pd.DataFrame([{"session": s["session"], "unit": k, "spikes": len(sp), "isi_sorted": rm.isi_fraction(sp),
                     "isi_as_stored": rm.isi_fraction(sp, sort=False)} for s in sessions for k, sp in enumerate(s["spikes"])])
print("units:", len(isi))
for c in ["isi_sorted", "isi_as_stored"]:
    print(f"{c}: below 3% {int((isi[c] < 0.03).sum())} ({100 * (isi[c] < 0.03).mean():.1f}%); median {100 * isi[c].median():.2f}%; max {100 * isi[c].max():.1f}%")
for thr in (0.03, 0.05, 0.10):
    print(f"units at or above {thr:.0%} (sorted): {int((isi.isi_sorted >= thr).sum())}")
print(isi[isi.isi_sorted >= 0.03].round(4).to_string(index=False))
print("by site, fraction below 3%:", isi.assign(site=isi.session.map({s['session']: s['site'] for s in sessions})).groupby("site").isi_sorted.apply(lambda v: round((v < 0.03).mean(), 3)).to_dict())
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(100 * isi.isi_sorted.clip(upper=0.2), bins=50)
ax.axvline(3, color="k"); ax.set_xlabel("% of ISIs < 3 ms (clipped at 20%)"); ax.set_ylabel("units")
plt.tight_layout(); plt.show()
print("units above 20% (clipped into the last bin):", int((isi.isi_sorted > 0.2).sum()))

**Result: partially reproduced.** 1858 of 1864 units (99.7%) have fewer than 3% of their interspike
intervals below 3 ms, above the 95% the criterion sets for a partial result. Six units are at or
above 3%, two of them above 20%. With the spike times as stored (unsorted in 13 units), 1860 units
pass.

## fig5a-mtl-locations

> All neurons were recorded from the medial temporal lobe, in the amygdala and the hippocampus.

In [ ]:
loc = pd.Series([s["electrodes"].location.values[e] for s in sessions for e in s["unit_electrode"]])
print("unit locations:", loc.value_counts().to_dict())
print("units in amygdala or hippocampus:", int(loc.str.contains("Amygdala|Hippocampus").sum()), "of", len(loc))

**Result: reproduced.** All 1864 units are on electrodes located in the left or right amygdala or
hippocampus.

## fig5c-auc-all

> Subjects distinguished new from old images; the behavioural AUC averaged over all sessions is
> 0.74 ± 0.10.

In [ ]:
beh = []
for s in sessions:
    r = rm.recognition(s)
    lab_, resp = r.label.values, r.response.values
    hi_f, lo_f = rm.confidence_accuracy(lab_, resp, "fixed")
    hi_d, lo_d = rm.confidence_accuracy(lab_, resp, "dynamic")
    beh.append({"session": s["session"], "site": s["site"], "auc": rm.auc(lab_, resp), "zroc_slope": rm.zroc_slope(lab_, resp),
                "acc_high_fixed": hi_f, "acc_low_fixed": lo_f, "acc_high_dynamic": hi_d, "acc_low_dynamic": lo_d,
                "hit_rate": np.mean(resp[lab_ == rm.OLD] >= 4), "fa_rate": np.mean(resp[lab_ == rm.NEW] >= 4)})
beh = pd.DataFrame(beh)
print(f"AUC over {len(beh)} sessions: mean {beh.auc.mean():.3f}, SD {beh.auc.std(ddof=1):.3f} (paper 0.74 ± 0.10)")
print(f"range {beh.auc.min():.3f} to {beh.auc.max():.3f}; sessions below 0.6: {int((beh.auc < 0.6).sum())}; at or below 0.5: {int((beh.auc <= 0.5).sum())}")
print(f"chance AUC is 0.5; one-sample t-test of AUC against 0.5: t = {stats.ttest_1samp(beh.auc, 0.5).statistic:.1f}, p = {stats.ttest_1samp(beh.auc, 0.5).pvalue:.2g}")
print(f"hit rate {beh.hit_rate.mean():.3f}, false alarm rate {beh.fa_rate.mean():.3f} (means over sessions)")
once = beh[~beh.session.isin(DUPLICATE_SECOND)]
print(f"variant, sessions 44 and 55 dropped: mean {once.auc.mean():.3f}, SD {once.auc.std(ddof=1):.3f}")
beh["auc_stated_coding"] = [rm.auc(1 - rm.recognition(s).label.values, rm.recognition(s).response.values) for s in sessions]
print(f"variant, label coding as the paper states (0 = old), recomputed: mean AUC {beh.auc_stated_coding.mean():.3f}, "
      f"range {beh.auc_stated_coding.min():.3f} to {beh.auc_stated_coding.max():.3f}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(beh.auc, bins=20); ax.set_xlabel("AUC"); ax.set_ylabel("sessions"); plt.tight_layout(); plt.show()

**Result: reproduced.** The AUC over 87 sessions is 0.753 with a standard deviation of 0.094, both
within tolerance of 0.74 ± 0.10, and well above the chance level of 0.5 (p = 2.2e-41). Recomputed with
the label coding the paper states, the mean AUC is 0.247 (0.081 to 0.555), below chance.

## fig5c-auc-sites

> Performance was similar at the two study sites, with an average AUC of 0.74 ± 0.09 for CS
> sessions and 0.69 ± 0.13 for TWH sessions.

In [ ]:
g = beh.groupby("site").auc.agg(["size", "mean", lambda v: v.std(ddof=1)]).rename(columns={"<lambda_0>": "sd"})
print(g.round(3).to_string())
print("paper: CS 0.74 ± 0.09, TWH 0.69 ± 0.13")
w = stats.ttest_ind(beh[beh.site == "CS"].auc, beh[beh.site == "TWH"].auc, equal_var=False)
print(f"CS against TWH, Welch's t-test: t = {w.statistic:.2f}, p = {w.pvalue:.3f}; difference of means {g.loc['CS', 'mean'] - g.loc['TWH', 'mean']:.3f}")
print("TWH sessions:", beh[beh.site == "TWH"][["session", "auc"]].round(3).to_dict("records"))

**Result: reproduced.** CS sessions have an AUC of 0.760 ± 0.085 (78 sessions) and TWH sessions
0.688 ± 0.138 (9 sessions), all four values within tolerance. The sites do not differ significantly
(Welch's t-test p = 0.161).

## fig5b-roc-asymmetric

> The ROC curve is asymmetric (p < 0.05), as expected for declarative memory.

In [ ]:
sl = beh.zroc_slope
t1 = stats.ttest_1samp(sl, 1.0)
wx = stats.wilcoxon(sl - 1.0)
print(f"z-ROC slope over {len(sl)} sessions: mean {sl.mean():.3f}, median {sl.median():.3f}, SD {sl.std(ddof=1):.3f}, range {sl.min():.2f} to {sl.max():.2f}")
print(f"one-sample t-test against 1: t = {t1.statistic:.2f}, p = {t1.pvalue:.2g}; sessions with slope below 1: {int((sl < 1).sum())} of {len(sl)}")
print(f"variant, Wilcoxon signed-rank against 1: W = {wx.statistic:.0f}, p = {wx.pvalue:.4g} (t-test p = {t1.pvalue:.4g})")
pooled_lab = np.concatenate([rm.recognition(s).label.values for s in sessions])
pooled_resp = np.concatenate([rm.recognition(s).response.values for s in sessions])
print(f"variant, slope of the pooled ROC (all trials of all sessions): {rm.zroc_slope(pooled_lab, pooled_resp):.3f}")
once = beh[~beh.session.isin(DUPLICATE_SECOND)].zroc_slope
print(f"variant, sessions 44 and 55 dropped: mean {once.mean():.3f}, p = {stats.ttest_1samp(once, 1.0).pvalue:.2g}")
print("\nper session: AUC, z-ROC slope, hit and false alarm rates")
print(beh[["session", "site", "auc", "zroc_slope", "hit_rate", "fa_rate"]].round(3).to_string(index=False))

**Result: reproduced.** The z-ROC slope is below 1 in 70 of 87 sessions, with a mean of 0.726
(t-test against 1, p = 1.6e-09). The Wilcoxon test gives p = 1.586e-09 (the t-test 1.631e-09), and the pooled
ROC has a slope of 0.816.

## fig5d-confidence-accuracy

> Accuracy is significantly higher in high than in low confidence trials (P < 0.001, paired t
> test) across all sessions.

In [ ]:
for split in ("fixed", "dynamic"):
    h, l = beh[f"acc_high_{split}"], beh[f"acc_low_{split}"]
    ok = h.notna() & l.notna()
    tr = stats.ttest_rel(h[ok], l[ok])
    tu = stats.ttest_ind(h[ok], l[ok])
    print(f"({'a' if split == 'fixed' else 'b'}) {split} split: {int(ok.sum())} sessions, {int((~ok).sum())} left out (no responses in a group): "
          f"{beh[~ok].session.tolist()}")
    print(f"    high {h[ok].mean():.1f}% against low {l[ok].mean():.1f}% (means); high > low in {int((h[ok] > l[ok]).sum())} sessions")
    print(f"    paired t-test: t = {tr.statistic:.2f}, p = {tr.pvalue:.2g};  variant, unpaired (the code's ttest2): p = {tu.pvalue:.2g}")
    print(f"    mean difference {np.mean(h[ok] - l[ok]):.1f} points, 2.5th to 97.5th percentile of the per-session differences {np.percentile(h[ok] - l[ok], [2.5, 97.5]).round(1)}")

**Result: reproduced.** Under the fixed split, accuracy is 80.1% for high and 60.9% for low confidence
over 84 sessions (paired t-test p = 3.2e-16); under the dynamic split 78.3% and 60.4% (p = 1.2e-13).
Three sessions without responses in one group are left out. The unpaired test the code runs also
gives p below 0.001.

## Single-unit selectivity

Every unit is tested on the recognition trials of its file. The firing rate in the window is the
spike count divided by the window length. Results for VS, MS and dual cells come from this one
computation; each claim's section prints its own numbers.

In [ ]:
rng_vs = np.random.default_rng(20200304)
rng_ms = np.random.default_rng(78)
rng_ms2 = np.random.default_rng(79)     # a second seed, to show how much the bootstrap's noise moves the counts
rng_null = np.random.default_rng(415)
N_NULL = 10
rows = []
started = time.time()
for s in sessions:
    r = rm.recognition(s)
    on, cat, lab_, resp = r.start_time.values, r.category.values, r.label.values, r.response.values
    correct = ((resp >= 4) & (lab_ == rm.OLD)) | ((resp <= 3) & (lab_ == rm.NEW))
    names = r.groupby("category").category_name.first()
    for k, sp in enumerate(s["spikes"]):
        row = {"session": s["session"], "site": s["site"], "unit": k,
               "channel": int(s["electrodes"].origChannel.values[s["unit_electrode"][k]]),
               "electrode_row": int(s["unit_electrode"][k]),
               "location": s["electrodes"].location.values[s["unit_electrode"][k]]}
        for wname, w in (("methods", rm.WINDOW_METHODS), ("code", rm.WINDOW_CODE)):
            x = rm.window_rates(sp, on, w)
            row[f"p_vs_{wname}"] = rm.anova_p(x, cat)
            row[f"p_ms_{wname}_all"] = rm.bootstrap_p(x[lab_ == rm.OLD], x[lab_ == rm.NEW], rng_ms)
            row[f"p_ms_{wname}_correct"] = rm.bootstrap_p(x[(lab_ == rm.OLD) & correct], x[(lab_ == rm.NEW) & correct], rng_ms)
            row[f"p_ms_{wname}_all_seed2"] = rm.bootstrap_p(x[lab_ == rm.OLD], x[lab_ == rm.NEW], rng_ms2)
            row[f"p_ms_{wname}_correct_seed2"] = rm.bootstrap_p(x[(lab_ == rm.OLD) & correct], x[(lab_ == rm.NEW) & correct], rng_ms2)
            row[f"best_{wname}"] = names[pd.Series(x).groupby(cat).mean().idxmax()]
        x = rm.window_rates(sp, on, rm.WINDOW_METHODS)
        row["null_vs"] = np.mean([rm.anova_p(x, rng_null.permutation(cat)) < 0.05 for _ in range(N_NULL)])
        shuffled = [rng_null.permutation(lab_) for _ in range(N_NULL)]
        row["null_ms"] = np.mean([rm.bootstrap_p(x[L == rm.OLD], x[L == rm.NEW], rng_null) < 0.05 for L in shuffled])
        row["rate_hz"] = len(sp) / (s["trials"].stop_time.max() - s["trials"].start_time.min())
        rows.append(row)
cells = pd.DataFrame(rows)
cells["vs"] = cells.p_vs_methods < 0.05
cells["ms"] = cells.p_ms_methods_all < 0.05
cells["once"] = ~cells.session.isin(DUPLICATE_SECOND)
print(f"{len(cells)} units tested in {time.time() - started:.0f} s; none dropped. Recognition trials per unit: 100 in every file.")
print(f"units with no spike in any recognition window (Methods' window): {int((cells.p_vs_methods == 1).sum())}")
print("correct trials per session (old, new):", sorted({(int((((rm.recognition(s).response >= 4) & (rm.recognition(s).label == 1))).sum()),
      int((((rm.recognition(s).response <= 3) & (rm.recognition(s).label == 0))).sum())) for s in sessions})[:5], "... (first five)")

## vs-cells-proportion

> 343 of the 1863 cells (18.4%) are visually selective, by a one-way ANOVA (P < 0.05) of the
> firing rate in recognition trials, in a 1 s window starting 200 ms after stimulus onset, across
> the 5 visual categories.

In [ ]:
def share(mask, base=None):
    base = cells if base is None else base
    return f"{int(mask.sum())} of {len(base)} ({100 * mask.mean():.1f}%)"
print("main (Methods' window, every unit):", share(cells.vs), " paper: 343 of 1863 (18.4%)")
print("variant, code's window 200-1700 ms:", share(cells.p_vs_code < 0.05))
print("variant, duplicated spike trains counted once:", share(cells[cells.once].vs, cells[cells.once]))
print(f"null, categories shuffled within session ({N_NULL} shuffles per unit): {100 * cells.null_vs.mean():.1f}% (expected 5%)")
for thr in (0.01, 0.05, 0.10):
    print(f"threshold p < {thr}: {100 * (cells.p_vs_methods < thr).mean():.1f}%")
print("\nby session (main), sessions with at least 20 units:")
bs = cells.groupby("session").agg(units=("vs", "size"), vs=("vs", "sum"))
print(bs[bs.units >= 20].assign(pct=lambda d: (100 * d.vs / d.units).round(1)).T.to_string())
print("range of per-session percentages (sessions with at least 20 units):", (100 * bs[bs.units >= 20].vs / bs[bs.units >= 20].units).min().round(1),
      "to", (100 * bs[bs.units >= 20].vs / bs[bs.units >= 20].units).max().round(1))
print("by area:", cells.groupby("location").vs.apply(lambda v: f"{int(v.sum())}/{len(v)}").to_dict())

**Result: reproduced.** With the Methods' window, 332 of 1864 units (17.8%) are visually selective,
within tolerance of 343 (18.4%), against 4.7% with shuffled categories. The code's window gives
343 of 1864 (18.4%), the paper's numbers. Counting duplicated spike trains once gives 18.0%.

## ms-cells-proportion

> 146 of the 1863 cells (7.8%) are memory selective, by a two-tailed bootstrap comparison of mean
> firing rates (1,000 runs, P < 0.05) between the 50 novel and 50 familiar recognition trials in a
> 1 s window starting 200 ms after stimulus onset.

In [ ]:
print("main (Methods' window, all 50 and 50 trials):", share(cells.ms), " paper: 146 of 1863 (7.8%)")
print("variant, Methods' window, correct trials only:", share(cells.p_ms_methods_correct < 0.05))
print("variant, code's window, all trials:", share(cells.p_ms_code_all < 0.05))
print("variant, code's choices (code's window, correct trials only):", share(cells.p_ms_code_correct < 0.05))
print("variant, duplicated spike trains counted once:", share(cells[cells.once].ms, cells[cells.once]))
print(f"null, labels shuffled within session ({N_NULL} shuffles per unit): {100 * cells.null_ms.mean():.1f}% (expected 5%)")
for thr in (0.01, 0.05, 0.10):
    print(f"threshold p < {thr}: {100 * (cells.p_ms_methods_all < thr).mean():.1f}%")
print("by site:", cells.groupby("site").ms.apply(lambda v: f"{int(v.sum())}/{len(v)}").to_dict())
print("by area:", cells.groupby("location").ms.apply(lambda v: f"{int(v.sum())}/{len(v)}").to_dict())
print("\nsecond bootstrap seed:")
for col, name in (("p_ms_methods_all", "main"), ("p_ms_methods_correct", "Methods' window, correct trials"),
                  ("p_ms_code_all", "code's window, all trials"), ("p_ms_code_correct", "code's choices")):
    a, b = cells[col] < 0.05, cells[col + "_seed2"] < 0.05
    print(f"  {name}: {int(a.sum())} and {int(b.sum())}; units selected under one seed only: {int((a != b).sum())}")
n, k, f0 = len(cells), int(cells.ms.sum()), cells.null_ms.mean()
lo0, hi0 = stats.binomtest(int(round(f0 * n * N_NULL)), n * N_NULL).proportion_ci()
print(f"\nnull fraction {100 * f0:.2f}% from {n * N_NULL} shuffled tests, 95% CI {100 * lo0:.2f}% to {100 * hi0:.2f}%")
print(f"main count {k} against the null: expected {n * f0:.1f}, binomial p (one-sided) = {stats.binomtest(k, n, f0, alternative='greater').pvalue:.3g}; "
      f"against 5%: expected {n * 0.05:.1f}, p = {stats.binomtest(k, n, 0.05, alternative='greater').pvalue:.3g}")
k2 = int((cells.p_ms_code_correct < 0.05).sum())
print(f"code's choices count {k2} against 5%: p = {stats.binomtest(k2, n, 0.05, alternative='greater').pvalue:.3g}")
ms_cols = ["p_ms_methods_all", "p_ms_methods_all_seed2", "p_ms_methods_correct", "p_ms_methods_correct_seed2",
           "p_ms_code_all", "p_ms_code_all_seed2", "p_ms_code_correct", "p_ms_code_correct_seed2"]
any_ms = (cells[ms_cols] < 0.05).any(axis=1)
print(f"\nper-unit p-values of every unit memory selective under some reading or seed ({int(any_ms.sum())} units):")
print(cells[any_ms][["session", "unit", "p_vs_methods", "p_vs_code"] + ms_cols].round(3).to_string(index=False))

**Result: not reproduced.** The test the Methods describe, all 50 new and 50 old trials in a 1 s
window, finds 110 of 1864 units (5.9%), outside the tolerance of 146 (7.8%). This is not
distinguishable from the 5.4% found with shuffled labels (one-sided binomial p = 0.18) and only just
above 5% (p = 0.044). A second bootstrap seed gives 118 (6.3%), also outside the tolerance. The code's
choices, correct trials only in a 1.5 s window, give 147 and 149 (7.9% and 8.0%) under the two seeds,
and correct trials in the Methods' window 143 and 140, all matching the paper. The outcome comes from
the Methods disagreeing with the paper's code on the trials used, not from the data: the paper's
number follows its code.

## dual-cells-proportion

> 26 cells (1.4%) are both visually and memory selective.

In [ ]:
dual = cells.vs & cells.ms
print("main:", share(dual), " paper: 26 of 1863 (1.4%)")
print("expected if VS and MS were independent:", f"{len(cells) * cells.vs.mean() * cells.ms.mean():.1f} units")
print("variant, code's choices:", share((cells.p_vs_code < 0.05) & (cells.p_ms_code_correct < 0.05)))
print("variant, duplicated spike trains counted once:", share(dual[cells.once], cells[cells.once]))
print("seed 2, main:", share(cells.vs & (cells.p_ms_methods_all_seed2 < 0.05)),
      "; code's choices:", share((cells.p_vs_code < 0.05) & (cells.p_ms_code_correct_seed2 < 0.05)))

**Result: not reproduced.** Under the Methods' tests 12 units (0.6%) are both VS and MS (14 with the
second bootstrap seed), outside the tolerance of 26 (1.4%) and below the 19.6 expected if the two
were independent. The code's choices give 26 (1.4%) under both seeds, the paper's numbers. As for
ms-cells-proportion, the outcome comes from the Methods disagreeing with the code, not from the
data.

## fig5f-site-proportions

> The percentages of VS, MS and dual cells are comparable between CS sessions (18.4%, 7.7% and
> 1.4%) and the new TWH sessions (19.4%, 10.4% and 1.3%).

In [ ]:
cells["dual"] = cells.vs & cells.ms
PAPER = {("vs", "CS"): 18.4, ("ms", "CS"): 7.7, ("dual", "CS"): 1.4, ("vs", "TWH"): 19.4, ("ms", "TWH"): 10.4, ("dual", "TWH"): 1.3}
within = 0
for kind in ("vs", "ms", "dual"):
    a = cells[cells.site == "CS"][kind]; b = cells[cells.site == "TWH"][kind]
    tab2 = [[int(a.sum()), int((~a).sum())], [int(b.sum()), int((~b).sum())]]
    p = stats.fisher_exact(tab2).pvalue
    for site, v in (("CS", a), ("TWH", b)):
        ok = abs(100 * v.mean() - PAPER[(kind, site)]) <= 0.1 * PAPER[(kind, site)]
        within += ok
        print(f"{kind:4s} {site:3s}: {int(v.sum())} of {len(v)} = {100 * v.mean():.1f}% (paper {PAPER[(kind, site)]}%, within 10%: {ok})")
    print(f"     Fisher's exact test CS against TWH: p = {p:.3f}")
print("values within tolerance:", within, "of 6")
code_sel = {"vs": cells.p_vs_code < 0.05, "ms": cells.p_ms_code_correct < 0.05}
code_sel["dual"] = code_sel["vs"] & code_sel["ms"]
for kind, sel in code_sel.items():
    print(f"variant, code's choices, {kind}:", cells.assign(x=sel).groupby("site").x.apply(lambda v: f"{int(v.sum())} of {len(v)} = {100 * v.mean():.1f}%").to_dict())
print("\n95% Clopper-Pearson intervals of the main percentages:")
for kind in ("vs", "ms", "dual"):
    for site in ("CS", "TWH"):
        v = cells[cells.site == site][kind]
        ci = stats.binomtest(int(v.sum()), len(v)).proportion_ci()
        print(f"  {kind:4s} {site:3s}: {100 * v.mean():.1f}% ({100 * ci.low:.1f} to {100 * ci.high:.1f})")

**Result: partially reproduced.** The sites do not differ by Fisher's exact test (VS p = 0.360, MS
p = 0.215, dual p = 1.000), but only 1 of the 6 site values is within tolerance under the Methods'
tests (CS VS 17.6%). The code's choices give CS 18.4%, 7.8% and 1.4% and TWH 19.5%, 10.4% and 1.3%,
the values of Figure 5f to within 0.1 point, so the partial outcome comes from the Methods
disagreeing with the code. TWH has 77 units, so its percentages rest on 17, 7 and 0 cells under the
main tests, with 95% intervals of 13.4% to 33.0%, 3.7% to 17.8% and 0.0% to 4.7%; the tests have
little power, and "no difference" carries little weight.

## twh-sessions

> The dataset includes 9 sessions from a new institution, Toronto Western Hospital.

In [ ]:
twh = info[info.site == "TWH"]
print("TWH sessions:", len(twh), twh.session.tolist(), "; subjects:", twh.subject.nunique(), "; units:", int(twh.units.sum()))
print("institution of these files:", twh.institution.unique().tolist())

**Result: reproduced.** Nine sessions (2001 to 2011) of 8 subjects come from Toronto Western Hospital,
with 77 units.

## fig4-example-vs-neuron

> Neuron 132-177-4-RA (session 132, channel 177, cell 4, right amygdala) is visually selective,
> responding most to images of animals.

In [ ]:
ex = cells[(cells.session == 132) & (cells.channel == 177)]
print(ex[["session", "channel", "electrode_row", "location", "p_vs_methods", "p_vs_code", "best_methods", "best_code"]].to_string(index=False))
s = by_id[132]; r = rm.recognition(s); k = int(ex.unit.iloc[0])
for wname, w in (("Methods' window", rm.WINDOW_METHODS), ("code's window", rm.WINDOW_CODE)):
    x = rm.window_rates(s["spikes"][k], r.start_time.values, w)
    print(wname, "mean rate (Hz) by category:", pd.Series(x).groupby(r.category_name.values).mean().round(2).to_dict())
print("units on channel 177 in session 132:", len(ex))

**Result: reproduced.** The single unit on channel 177 of session 132 sits in the right amygdala at
electrode row 4. It is category selective (ANOVA p = 1.2e-22 in the Methods' window, 7.9e-19 in the
code's) and fires most to animals (6.75 Hz against 0.25 to 0.85 Hz for the other categories).

## previous-release-link

> The sessions of the earlier release (Faraut et al. 2018) are found in this dandiset under the
> same subject, session id and task variant.

The earlier release's sessions come from Table 1 of Faraut et al. 2018 (doi:10.1038/sdata.2018.10),
copied into the helper module. The task variant of a file is read from its stimulus set, through the
variant Online-only Table 1 gives each stimulus set (shown under counts-sessions).

In [ ]:
norm = lambda v: v.str.replace("CS", "C").str.replace(r"^([A-Z])0*", r"\1", regex=True)
early = rm.faraut_2018_table_1()
early["subject_norm"] = norm(early.table_subject)
first_cat = pd.Series({s["session"]: sorted(s["trials"].category_name.unique())[0] for s in sessions})
cat_to_variant = pd.crosstab(first_cat.rename("cat"), pd.Series(first_cat.index.map(var), index=first_cat.index, name="variant")).idxmax(axis=1)
files = info[["session", "label", "age", "sex"]].assign(subject_norm=lambda d: norm(d.label),
                                                       variant_file=lambda d: d.session.map(first_cat).map(cat_to_variant).values)
lk = early.merge(files, on="session", how="left", suffixes=("_2018", "_file"))
lk["found"] = lk.label.notna()
lk["agrees"] = lk.found & (lk.subject_norm_2018 == lk.subject_norm_file) & (lk.variant == lk.variant_file)
print("sessions in Faraut et al. 2018 Table 1:", len(early), "; subjects:", early.table_subject.nunique())
print("found in the dandiset by session id:", int(lk.found.sum()), "of", len(lk), f"({100 * lk.found.mean():.1f}%)")
print("found and agreeing on subject and task variant:", int(lk.agrees.sum()), "of", len(lk), f"({100 * lk.agrees.mean():.1f}%)")
print("\nnot found or not agreeing:")
print(lk[~lk.agrees][["table_subject", "session", "variant", "label", "variant_file"]].to_string(index=False))
subj18 = lk[lk.found].groupby("table_subject").agg(age_2018=("age_2018", "first"), sex_2018=("sex_2018", "first"),
                                                 age_file=("age_file", "first"), sex_file=("sex_file", "first"))
subj18["age_file"] = subj18.age_file.astype(int)
same = (subj18.age_2018 == subj18.age_file) & (subj18.sex_2018 == subj18.sex_file)
print("\nmatched subjects with the same age and sex in the files as in the 2018 table:", int(same.sum()), "of", len(subj18))
print(subj18[~same].to_string())

**Result: partially reproduced.** 64 of the 65 sessions of Faraut et al. 2018 (98.5%) are in the
dandiset under the same subject and task variant. Session 98 of subject H48 is not. Age and sex
agree for 39 of the 42 matched subjects; C24, H33 and H48 differ, the same three subjects that
differ from Online-only Table 1, and in each the two papers' tables agree with each other.

## previous-release-additions

> Relative to the earlier release of this task's data (Faraut et al. 2018), this release adds 23
> sessions, 17 subjects and 288 neurons.

In [ ]:
new_ses = info[~info.session.isin(early.session)]
old_subj = set(norm(info[info.session.isin(early.session)].label))
new_subj = sorted(set(norm(new_ses.label)) - old_subj)
print("new sessions:", len(new_ses), new_ses.session.tolist(), " paper: 23")
print("new subjects:", len(new_subj), new_subj, " paper: 17")
print("units in the new sessions:", int(new_ses.units.sum()), " paper: 288")
print("  of which CS:", int(new_ses[new_ses.site == "CS"].units.sum()), "in", int((new_ses.site == "CS").sum()), "sessions;",
      "TWH:", int(new_ses[new_ses.site == "TWH"].units.sum()), "in", int((new_ses.site == "TWH").sum()), "sessions")
print("subjects of the new sessions that also have earlier sessions:", sorted(set(norm(new_ses.label)) & old_subj))
print("\nvariant, differences of the totals the two papers state (2020 minus 2018): sessions 87 - 65 =", 87 - 65,
      "; subjects 59 - 42 =", 59 - 42, "; neurons 1863 - 1576 =", 1863 - 1576)
print("units in the dandiset's sessions that are in the 2018 table:", int(info[info.session.isin(early.session)].units.sum()),
      " (Faraut et al. 2018: 1576 neurons in 65 sessions)")
print("Faraut et al. 2018 neurons plus the stated additions: 1576 + 288 =", 1576 + 288, "; unit rows in the dandiset:", int(info.units.sum()))

**Result: partially reproduced.** Against the session list of Faraut et al. 2018, the dandiset has 23
new sessions (14 CS, 9 TWH) from 17 new subjects, the paper's numbers. The new sessions hold 325
units, outside the tolerance of 288; the sessions also in the earlier release hold 1539, where
Faraut et al. 2018 gives 1576 neurons for its 65 sessions. The paper's 288 equals 1864 minus 1576,
which suggests it is a difference of totals, not a count of the new sessions. The missing session
98 (previous-release-link) does not change the new sessions or subjects; it may account for part
of the 37 units fewer in the earlier sessions.

## fig3-example-vs-neuron (blocked)

> Neuron 144-140-3-RH (session 144, channel 140, cell 3, right hippocampus) is visually selective
> for phones.

Session 144 is not in the dandiset (data check) or in Online-only Table 1, so the claim is blocked.
As a weak check of whether the neuron is in the dandiset under another session id, every unit on
original channel 140 in the right hippocampus is listed with its category ANOVA and preferred
category. This does not decide the claim. The paper's code was read for the same purpose, not run
here: its demo script `python/export/demo/export.py` sets `NOID = 144` under the comment
`#session: 'TWH103_111918'`, and its session list gives subject T103 the one session 2009.

In [ ]:
cand = cells[(cells.channel == 140) & (cells.location == "Right Hippocampus")]
print("units on channel 140 in the right hippocampus:", len(cand))
cand = cand.assign(cluster=[int(by_id[a]["cluster_id"][k]) for a, k in zip(cand.session, cand.unit)])
print(cand[["session", "unit", "electrode_row", "cluster", "p_vs_methods", "best_methods", "p_vs_code", "best_code"]].to_string(index=False))
print("of these, VS (Methods' window) and preferring a phone category:",
      int((cand.vs & cand.best_methods.str.contains("hone", case=False)).sum()))
print("category names containing 'phone':", sorted({n for s in sessions for n in s["trials"].category_name.unique() if "hone" in n.lower()}))
print("subject of session 2009:", by_id[2009]["label"], "; sessions of this subject:", info[info.label == by_id[2009]["label"]].session.tolist())

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| counts-neurons-subjects | reproduced |
| counts-sessions | reproduced |
| table1-demographics | partially reproduced |
| task-structure | partially reproduced |
| new-old-label-coding | not reproduced |
| ttl-codes | partially reproduced |
| experiment-ids | partially reproduced |
| session-dates | partially reproduced |
| units-table | partially reproduced |
| electrodes-table | not reproduced |
| isi-criterion | partially reproduced |
| fig5a-mtl-locations | reproduced |
| fig5c-auc-all | reproduced |
| fig5c-auc-sites | reproduced |
| fig5b-roc-asymmetric | reproduced |
| fig5d-confidence-accuracy | reproduced |
| vs-cells-proportion | reproduced |
| ms-cells-proportion | not reproduced |
| dual-cells-proportion | not reproduced |
| fig5f-site-proportions | partially reproduced |
| twh-sessions | reproduced |
| fig4-example-vs-neuron | reproduced |
| previous-release-link | partially reproduced (with outside input) |
| previous-release-additions | partially reproduced (with outside input) |
| fig3-example-vs-neuron | blocked: session 144 is not in the dandiset, and nothing maps it to a session there |

**How the analysis differs from the paper's.**

- Selectivity uses the Methods' window (200 to 1200 ms) and, for memory selectivity, all 50 new and
  50 old trials. The paper's code uses 200 to 1700 ms and correct trials only; with these choices
  the counts are 343 VS, 147 MS and 26 dual cells, the paper's numbers. The MS, dual and Figure 5f
  outcomes rest on the Methods' description, and come from its disagreement with the code.
- The bootstrap is the code's algorithm with a fixed seed; a second seed moves the main MS count
  from 110 to 118 and the code's choices from 147 to 149.
- The paper names no test for the ROC asymmetry or for "comparable" proportions; a t-test of the
  z-ROC slope against 1 and Fisher's exact test were chosen.
- Accuracy by confidence uses a paired test, as the Results say; the code runs an unpaired one,
  shown as a variant.
- Units are counted per file, as in the paper; counts with the duplicated spike trains of sessions
  44 and 55 dropped are shown.
- The earlier release's sessions are taken from Table 1 of Faraut et al. 2018, not from its Dryad
  archive, which a script could not fetch.

**What was found about the dandiset.**

- The new/old label is coded 1 = old, the reverse of the paper's text and the files' own column
  description. A user following the description would compute an AUC of 0.25.
- Sessions 43 and 44, and 54 and 55, hold identical spike trains: the same neurons are deposited
  under two sessions, so 55 of the 1864 units are repeats. Session 55 also repeats session 54's
  learning block.
- Seven sessions have 50 encoding trials and one has 97, where the paper describes 100; part of the
  images later shown as old are missing from their encoding blocks.
- Electrode impedance is empty for every electrode, coordinates are missing for 99 electrodes, the
  isolation distance is NaN for 493 units, and SNR is not finite for 86.
- Spike times are unsorted in 13 units, and two units of session 58 hold each spike time twice.
- Age or sex differ from Online-only Table 1 for three subjects. One session date keeps its day, and
  another is set to 2000-01-01.
- Event codes outside Table 1 occur in nine files and experiment ids outside Table 2 in four.
- The Methods give 89 sessions where the paper's table and the files have 87, and the unit count is
  1864 where the paper gives 1863.
- The paper's Figure 3 example neuron is from session 144, which is not in the dandiset. The
  paper's code names NOID 144 as a recording of subject T103, whose one session here is 2009; one
  unit of session 2009 on channel 140 in the right hippocampus prefers phones, but no record maps
  144 to 2009.
- Session 98 of the earlier release (subject H48) is not in the dandiset. The new sessions hold 325
  units, not the 288 the paper gives.